# Customer Intelligence & Support Copilot



## 0. Repository map and end-to-end flow

`generate_data.py` → `data/raw/*.csv` → audit → clean → `data/clean/*.csv` + `data/instamart.db` → RFM/segments + lapse model → privacy-safe ticket triage → policy retrieval + refund tools.

Teaching point: **fetching data** does not always mean calling an API. Here, the source is a local CSV export. In a production system, the same ingestion layer could use an approved API or warehouse connector, but credentials and network access are intentionally out of scope.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "generate_data.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)
print("Root files:", sorted(p.name for p in ROOT.iterdir() if p.is_file()))

Project root: c:\instamart_customer_copilot
Root files: ['.gitignore', 'GITHUB_RESOURCES.md', 'README.md', 'generate_data.py', 'mock_llm.py', 'requirements.txt', 'run_project.py']


## 1. Generate / fetch the raw data

The data generator is deterministic (fixed random seed). It writes customer, product, order, support-ticket, evaluation-question, agent-scenario inputs and policy text files.


In [9]:
import subprocess
import sys

generator = ROOT / "generate_data.py"

result = subprocess.run(
    [sys.executable, str(generator)],
    cwd=ROOT,
    check=True,
    capture_output=True,
    text=True
)

print("Generator output:")
print(result.stdout)

raw_dir = ROOT / "data" / "raw"

print("\nDoes data/raw exist?", raw_dir.exists())

if raw_dir.exists():
    print("\nGenerated raw files:")
    for file in sorted(raw_dir.iterdir()):
        print(file.name)
else:
    print("\nThe raw folder was not created.")
    print("Project folders:", [p.name for p in ROOT.iterdir() if p.is_dir()])

Generator output:
generated: customers=1200 products=40 orders_raw=13084 tickets=164


Does data/raw exist? True

Generated raw files:
customers.csv
orders.csv
policy_eval.csv
products.csv
tickets.csv


In [11]:
data_dir = ROOT / "data"

print("DATA FOLDER STRUCTURE\n")

for path in sorted(data_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(ROOT))

DATA FOLDER STRUCTURE

data\policies\delivery_policy.txt
data\policies\escalation_policy.txt
data\policies\payment_policy.txt
data\policies\privacy_policy.txt
data\policies\refund_policy.txt
data\raw\customers.csv
data\raw\orders.csv
data\raw\policy_eval.csv
data\raw\products.csv
data\raw\tickets.csv


## 2. Read the exports and understand their shape

Use `keep_default_na=False` for ticket text so empty strings remain empty strings. This is important because empty tickets must be identified by an explicit guardrail rather than silently changing the input representation.

In [10]:
import pandas as pd
customers_raw = pd.read_csv(raw_dir / "customers.csv", keep_default_na=False)
products_raw = pd.read_csv(raw_dir / "products.csv", keep_default_na=False)
orders_raw = pd.read_csv(raw_dir / "orders.csv", keep_default_na=False)
tickets_raw = pd.read_csv(raw_dir / "tickets.csv", keep_default_na=False)

for name, df in [("customers", customers_raw), ("products", products_raw),
                 ("orders", orders_raw), ("tickets", tickets_raw)]:
    print(f"\\n{name}: shape={df.shape}")
    display(df.head(3))
    display(pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "blank_count": df.astype(str).apply(lambda col: col.str.strip().eq("").sum()),
        "missing_pct": (df.isna().mean() * 100).round(2)
    }))

\ncustomers: shape=(1200, 5)


,customer_id,city,signup_date,age,gender
0,C0001,Chennai,2024-01-13,35,MALE
1,C0002,DELHI,2024-02-17,31,MALE
2,C0003,bengaluru,2025-02-18,18,F


,dtype,missing_count,blank_count,missing_pct
customer_id,str,0,0,0.0
city,str,0,0,0.0
signup_date,str,0,0,0.0
age,str,0,61,0.0
gender,str,0,30,0.0


\nproducts: shape=(40, 4)


,product_id,product_name,category,price_inr
0,P001,Basmati Rice 1kg,Grocery,129
1,P002,Toor Dal 1kg,Grocery,165
2,P003,Atta 5kg,Grocery,245


,dtype,missing_count,blank_count,missing_pct
product_id,str,0,0,0.0
product_name,str,0,0,0.0
category,str,0,0,0.0
price_inr,int64,0,0,0.0


... (2 outputs left out)

,dtype,missing_count,blank_count,missing_pct
order_id,str,0,0,0.0
customer_id,str,0,0,0.0
product_id,str,0,0,0.0
quantity,int64,0,0,0.0
order_ts,str,0,0,0.0
status,str,0,0,0.0
discount_pct,int64,0,0,0.0
order_amount,str,0,0,0.0
delivery_minutes,str,0,2541,0.0


\ntickets: shape=(164, 5)


,ticket_id,customer_id,order_id,text,true_category
0,T0024,C0486,ORD-105035,There is an unauthorized transaction on my car...,fraud
1,T0073,C0080,ORD-100816,The invoice for ORD-100816 shows a wrong amoun...,billing
2,T0095,C1087,ORD-111156,The payment page shows an error and the app fr...,technical


,dtype,missing_count,blank_count,missing_pct
ticket_id,str,0,0,0.0
customer_id,str,0,0,0.0
order_id,str,0,8,0.0
text,str,0,4,0.0
true_category,str,0,0,0.0


### Audit questions for students
- Which columns are supposed to be unique keys?
- Are duplicates exact duplicates or multiple states of the same order?
- Which values are impossible (negative delivery duration, non-numeric money, implausible age)?
- Why should we distinguish an empty string from a missing value?
- What would go wrong if we modelled before cleaning?

In [18]:
from src.data_pipeline import audit, clean_data, validate
audit()


customers: shape=(1200, 5)
customer_id    str
city           str
signup_date    str
age            str
gender         str
dtype: object
             missing_count  missing_pct
customer_id              0          0.0
city                     0          0.0
signup_date              0          0.0
age                      0          0.0
gender                   0          0.0

products: shape=(40, 4)
product_id        str
product_name      str
category          str
price_inr       int64
dtype: object
              missing_count  missing_pct
product_id                0          0.0
product_name              0          0.0
category                  0          0.0
price_inr                 0          0.0

orders: shape=(13084, 9)
order_id              str
customer_id           str
product_id            str
quantity            int64
order_ts              str
status                str
discount_pct        int64
order_amount          str
delivery_minutes      str
dtype: object
                 

## 3. Clean the data and write trusted outputs

The pipeline applies explicit rules:
- standardize city and gender;
- make impossible ages missing and fill with city median;
- parse timestamps, remove exact duplicates, then keep the latest row per order ID;
- coerce invalid amounts to missing and calculate replacements from product price × quantity × discount;
- set negative delivery times to missing and impute only completed orders using product-category medians;
- save clean CSVs and SQLite tables.

Ask students to explain why cancelled orders should keep a missing delivery duration.

In [ ]:
clean = clean_data()
for name, df in clean.items():
    print(name, df.shape)
    display(df.head(3))
validate()

customers (1200, 5)


,customer_id,city,signup_date,age,gender
0,C0001,Chennai,2024-01-13,35,male
1,C0002,Delhi,2024-02-17,31,male
2,C0003,Bengaluru,2025-02-18,18,female


products (40, 4)


,product_id,product_name,category,price_inr
0,P001,Basmati Rice 1kg,Grocery,129
1,P002,Toor Dal 1kg,Grocery,165
2,P003,Atta 5kg,Grocery,245


orders (12309, 9)


,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes
827,ORD-100001,C0001,P001,1,2025-09-16 08:07:21,completed,15,109.65,20
4294,ORD-100002,C0001,P008,1,2025-09-03 07:34:01,completed,40,25.20,23
9973,ORD-100003,C0001,P008,1,2025-04-08 17:08:14,completed,0,42.00,23


ALL CHECKPOINTS PASSED


## 4. Query the SQLite database

The clean tables are persisted in `data/instamart.db`. SQLite gives us a portable analytical database that can be queried without setting up a server.

Run the supplied SQL examples and compare SQL totals to pandas results.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np

# Connect to the SQLite database
db_path = ROOT / "data" / "instamart.db"

with sqlite3.connect(db_path) as con:

    # Load the cleaned tables
    customers = pd.read_sql_query("SELECT * FROM customers", con)
    products = pd.read_sql_query("SELECT * FROM products", con)
    orders = pd.read_sql_query("SELECT * FROM orders", con)

# 1. PANDAS CROSS-CHECK: REBUILD SQL Q2

# Keep only completed orders
completed_orders = orders[orders["status"] == "completed"].copy()

# Join orders with customers and products
analysis_df = (
    completed_orders
    .merge(customers[["customer_id", "city"]], on="customer_id")
    .merge(products[["product_id", "category"]], on="product_id")
)

# Create a pivot table with row, column and grand totals
revenue_pivot = pd.pivot_table(
    analysis_df,
    index="city",
    columns="category",
    values="order_amount",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="Grand Total"
)

print("PANDAS PIVOT TABLE")
display(revenue_pivot.round(2))

# Compare Pandas grand total with SQL Q2
pandas_total = revenue_pivot.loc["Grand Total", "Grand Total"]

with sqlite3.connect(db_path) as con:
    sql_q2 = pd.read_sql_query(
        """
        SELECT
            c.city,
            p.category,
            ROUND(SUM(o.order_amount), 2) AS revenue
        FROM orders AS o
        JOIN customers AS c
            ON o.customer_id = c.customer_id
        JOIN products AS p
            ON o.product_id = p.product_id
        WHERE o.status = 'completed'
        GROUP BY c.city, p.category
        """,
        con
    )

sql_total = sql_q2["revenue"].sum()

print("Pandas Grand Total INR:", round(pandas_total, 2))
print("SQL Grand Total INR:", round(sql_total, 2))
print("Totals Match:", np.isclose(pandas_total, sql_total, atol=0.01))

# 2. CITY-WISE REVENUE USING GROUPBY

city_summary = (
    completed_orders
    .merge(customers[["customer_id", "city"]], on="customer_id")
    .groupby("city")
    .agg(
        total_revenue=("order_amount", "sum"),
        average_order_amount=("order_amount", "mean"),
        order_count=("order_amount", "count")
    )
    .reset_index()
    .sort_values("city")
)

city_summary["total_revenue"] = city_summary["total_revenue"].round(2)
city_summary["average_order_amount"] = city_summary["average_order_amount"].round(2)

print("\nCITY-WISE COMPLETED ORDER SUMMARY")
display(city_summary)

# 3. ORDER AMOUNT DISTRIBUTION AND OUTLIERS

amount = completed_orders["order_amount"]

# Basic statistics
mean_amount = amount.mean()
median_amount = amount.median()
skewness = amount.skew()

# Quartiles and Interquartile Range
q1 = amount.quantile(0.25)
q2 = amount.quantile(0.50)
q3 = amount.quantile(0.75)

iqr = q3 - q1

# Lower and upper outlier fences
lower_fence = q1 - (1.5 * iqr)
upper_fence = q3 + (1.5 * iqr)

# Count outliers
lower_outliers = (amount < lower_fence).sum()
upper_outliers = (amount > upper_fence).sum()

distribution_summary = pd.DataFrame({
    "Metric": [
        "Mean",
        "Median",
        "Skewness",
        "Q1 (25%)",
        "Q2 (50%)",
        "Q3 (75%)",
        "IQR",
        "Lower Fence",
        "Upper Fence",
        "Outliers Below Lower Fence",
        "Outliers Above Upper Fence"
    ],
    "Value": [
        round(mean_amount, 2),
        round(median_amount, 2),
        round(skewness, 2),
        round(q1, 2),
        round(q2, 2),
        round(q3, 2),
        round(iqr, 2),
        round(lower_fence, 2),
        round(upper_fence, 2),
        int(lower_outliers),
        int(upper_outliers)
    ]
})

print("\nORDER AMOUNT DISTRIBUTION")
display(distribution_summary)


# 4. CORRELATION MATRIX

correlation_columns = [
    "order_amount",
    "discount_pct",
    "quantity",
    "delivery_minutes"
]

# Convert columns to numeric values
correlation_data = completed_orders[correlation_columns].apply(
    pd.to_numeric, errors="coerce"
)

# Calculate correlation matrix
correlation_matrix = correlation_data.corr()

print("\nCORRELATION MATRIX")
display(correlation_matrix.round(3))

# Find the variable pair with the largest absolute correlation
upper_triangle = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)

strongest_pair = upper_triangle.abs().stack().idxmax()
strongest_value = upper_triangle.loc[strongest_pair[0], strongest_pair[1]]

print("\nSTRONGEST CORRELATION PAIR")
print("Variables:", strongest_pair)
print("Correlation:", round(strongest_value, 3))
print("Absolute correlation:", round(abs(strongest_value), 3))


PANDAS PIVOT TABLE


category,Dairy,Electronics,Grocery,Personal Care,Snacks,Grand Total
city,,,,,,
Bengaluru,87075.05,233909.45,110915.65,20193.85,71549.40,523643.40
Chennai,78301.80,195508.80,93817.45,27675.40,68325.70,463629.15
Delhi,117271.70,158549.10,112658.05,23471.50,90879.45,502829.80
Hyderabad,83731.00,228913.10,109475.40,24800.05,81670.35,528589.90
Mumbai,75209.85,214738.40,78970.15,38018.00,72122.10,479058.50
Pune,90349.55,187629.25,108456.60,34275.15,80207.50,500918.05
Grand Total,531938.95,1219248.10,614293.30,168433.95,464754.50,2998668.80


Pandas Grand Total INR: 2998668.8
SQL Grand Total INR: 2998668.8
Totals Match: True

CITY-WISE COMPLETED ORDER SUMMARY


,city,total_revenue,average_order_amount,order_count
0,Bengaluru,523643.40,276.47,1894
1,Chennai,463629.15,272.72,1700
2,Delhi,502829.80,243.62,2064
3,Hyderabad,528589.90,284.95,1855
4,Mumbai,479058.50,297.00,1613
5,Pune,500918.05,252.48,1984



ORDER AMOUNT DISTRIBUTION


,Metric,Value
0,Mean,269.91
1,Median,135.00
2,Skewness,4.61
3,Q1 (25%),64.80
4,Q2 (50%),135.00
5,Q3 (75%),297.00
6,IQR,232.20
7,Lower Fence,-283.50
8,Upper Fence,645.30
9,Outliers Below Lower Fence,0.00



CORRELATION MATRIX


,order_amount,discount_pct,quantity,delivery_minutes
order_amount,1.000,-0.095,0.087,-0.003
discount_pct,-0.095,1.000,-0.000,-0.010
quantity,0.087,-0.000,1.000,-0.013
delivery_minutes,-0.003,-0.010,-0.013,1.000



STRONGEST CORRELATION PAIR
Variables: ('order_amount', 'discount_pct')
Correlation: -0.095
Absolute correlation: 0.095


## 5. Part B — customer segments and lapse prediction

RFM means:
- **Recency:** days since the most recent completed order at the snapshot date.
- **Frequency:** number of distinct completed orders.
- **Monetary:** total spend across completed orders.

The model module scales RFM for K-means, compares K values, uses PCA to inspect compressed variance, and builds a lapse model. The lapse model's input features use only orders before 2025-09-01; September is reserved to define the label. The scaler and encoder are fitted inside a training pipeline, not on the test data.

In [ ]:
from src.customer_models import rfm_and_segments, build_lapse_model
rfm, k_diagnostics, segment_profile, pca_variance = rfm_and_segments(k=4)
display(k_diagnostics.round(3))
display(segment_profile.round(2))
print("PCA explained variance ratio:", pca_variance, "total:", pca_variance.sum())
display(rfm.head())
lapse_summary = build_lapse_model()
lapse_summary

Inertia generally decreases as the number of clusters increases because more clusters allow customers to be grouped more closely around their cluster centres. However, choosing K=8 simply because it has lower inertia may create too many small, less meaningful segments. Therefore, we should also consider the elbow method, silhouette score, and business interpretability to select the most suitable K.

Ask students to identify:
1. Which dates can enter the lapse-model features?
2. Why must September orders not enter the features?
3. Why is a Premium-only retention list labelled *illustrative* if the Premium flag comes from whole-window RFM?
4. Why can accuracy be misleading when a substantial minority of customers lapse?

### business names and segmentation 

## Task 4: Lapse Prediction

--Features: Use orders before 1 September 2025 only.
--Target label: Check whether the customer placed any completed order from 1–30 September 2025.
--Cohort: Customers with at least one completed order before 1 September 2025.

In [ ]:
rfm, diagnostics, profile, pca_variance = rfm_and_segments(k=4)

display(diagnostics)
display(profile)

print("PCA explained variance:", pca_variance)

## 6. Part C — support ticket triage

This pipeline demonstrates:
1. input guardrails for empty and overlong tickets;
2. ordered PII redaction (email → phone → order ID);
3. retry logic for simulated transient errors;
4. strict category validation and a keyword fallback;
5. token budgeting and human-review routing.

The exact user message sent to the mock classifier is the redacted ticket text only.

In [ ]:
from src.triage import run_pipeline, FEW, ZERO
triage_results, triage_summary = run_pipeline(prompt=FEW, token_budget=100000)
print(triage_summary)
display(triage_results.head(10))
display(triage_results.status.value_counts().rename_axis("status").reset_index(name="tickets"))

### Discussion: privacy and safety
- Redaction is a defence-in-depth step, not a guarantee that every sensitive detail is removed.
- Why are invalid categories sent to human review instead of guessed?
- Why should fraud override ordinary customer-segment routing?
- Why do empty/long tickets need to be blocked before any model call?

## 7. Part D — policy RAG

The retrieval layer chunks policy documents, fits a TF-IDF vectorizer and ranks chunks by cosine similarity. It passes retrieved chunks to the offline policy assistant, which should answer only from the context and fall back when a question is not covered.

In [ ]:
from src.policy_rag import PolicyRAG, evaluate
rag = PolicyRAG(chunk_size=40, overlap=10)
print("Policy chunks:", len(rag.chunks))
display(pd.DataFrame(rag.chunks).head())
print(rag.answer("What delivery time makes a completed order eligible for a full refund?"))
policy_eval = evaluate(rag)
display(policy_eval)
print("Expected-document hits:", int(policy_eval.retrieved_doc.sum()))
print("Expected-fact hits:", int(policy_eval.fact_found.sum()))
print("Fallbacks:", int(policy_eval.fallback.sum()))

## 8. Part D — refund tools and safe stopping

The refund agent demonstrates a tool loop: look up an order in SQLite, retrieve policy, calculate the eligible refund, then issue it or escalate to a human. In this teaching kit the ledger is a local JSONL file; no money moves.

Before running agent demonstrations, inspect `src/refund_agent.py` and ask students to trace how a tool name maps to a Python function.

In [ ]:
from src.refund_agent import get_order, search_policy, calculate_refund, TOOL_MAP, run_agent
print("Tools:", sorted(TOOL_MAP))
print("Example order lookup:")
with sqlite3.connect(db_path) as con:
    example_order = pd.read_sql_query(
        "SELECT order_id FROM orders WHERE status='completed' AND delivery_minutes > 45 LIMIT 1", con
    ).iloc[0, 0]
print(example_order, get_order(example_order))
print("Refund calculation:", calculate_refund(250.0, 52))
print("Timeout safety demo:", run_agent(f"Refund order {example_order}", scenario_id="timeout-demo", time_limit_seconds=0))

## 9. Final generated artifacts and reproducibility check

Review the CSVs and model file generated by the pipeline. In class, have students delete `data/clean/`, `data/instamart.db`, and `outputs/`, then rerun the documented commands to verify that the project can be rebuilt from source.

**Important:** This teaching kit is an instructional implementation. If the course grades exact numeric checkpoints from the original brief, run every checkpoint and reconcile differences; do not hardcode expected answers.

In [ ]:
for folder in [ROOT/"data"/"clean", ROOT/"outputs", ROOT/"state"]:
    print(f"\\n{folder.relative_to(ROOT)}")
    print(sorted(p.name for p in folder.iterdir()) if folder.exists() else "(not created)")
print("Database exists:", db_path.exists())